In [25]:
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import torchvision.models as models
import os, cv2, random, numpy as np
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms


In [26]:
def corrupt_image(img):
    def add_noise(i):
        return np.clip(i+np.random.normal(0,20,i.shape),0,255).astype(np.uint8)
    def blur(i):
        return cv2.GaussianBlur(i,(5,5),0)
    def lowres(i):
        h,w=i.shape[:2]
        return cv2.resize(cv2.resize(i,(w//3,h//3)),(w,h))
    return random.choice([add_noise,blur,lowres])(img)


class RAImageDataset(Dataset):
    def __init__(self, root_dir, img_size=224):
        self.paths=[os.path.join(root_dir,f) for f in os.listdir(root_dir)]

        self.transform=transforms.Compose([
            transforms.ToTensor(),
            transforms.Resize((img_size,img_size))
        ])

    def __len__(self): return len(self.paths)

    def __getitem__(self,idx):
        img=cv2.imread(self.paths[idx])
        img=cv2.cvtColor(img,cv2.COLOR_BGR2RGB)

        corrupted=corrupt_image(img.copy())

        return self.transform(corrupted), self.transform(img)


In [27]:
DATA_PATH = r"D:\Image Recognition\data"
dataset = RAImageDataset(DATA_PATH)
loader = DataLoader(dataset,batch_size=8,shuffle=True)


In [28]:
class Block(nn.Module):
    def __init__(self,a,b):
        super().__init__()
        self.net=nn.Sequential(
            nn.Conv2d(a,b,3,1,1),
            nn.BatchNorm2d(b),
            nn.ReLU())
    def forward(self,x): return self.net(x)

class ProcessingNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(
            Block(3,64),Block(64,64),
            nn.MaxPool2d(2),
            Block(64,128),Block(128,128),
            nn.Upsample(scale_factor=2),
            Block(128,64),
            nn.Conv2d(64,3,3,1,1),
            nn.Sigmoid())
    def forward(self,x): return self.net(x)


In [29]:
device="cuda" if torch.cuda.is_available() else "cpu"

teacher = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
teacher = teacher.to(device)
teacher.eval()

# classifier normalization
normalize = transforms.Normalize(
    mean=[0.485,0.456,0.406],
    std=[0.229,0.224,0.225]
)
print(device)


cuda


In [30]:
model=ProcessingNet().to(device)
model.load_state_dict(torch.load("baseline_model.pth"))


<All keys matched successfully>

In [31]:
image_loss_fn = nn.MSELoss()
class_loss_fn = nn.CrossEntropyLoss()

# lambda_values = [0, 0.01, 0.1, 0.5, 1]

optimizer = optim.Adam(model.parameters(),lr=1e-4)


In [32]:
lambda_values = [0, 0.01, 0.1, 0.5, 1]

for lambda_recog in lambda_values:

    print(f"\n========== Training with λ = {lambda_recog} ==========\n")



    for epoch in range(5):

        total_loss = 0
        pbar = tqdm(loader)

        for corrupted, clean in pbar:

            corrupted = corrupted.to(device)
            clean = clean.to(device)

            # ----- Forward -----
            restored = model(corrupted)

            # ----- Image Loss -----
            img_loss = image_loss_fn(restored, clean)

            # ----- Recognition Loss -----
            with torch.no_grad():
                teacher_pred = teacher(normalize(clean))
                labels = teacher_pred.argmax(dim=1)

            pred = teacher(normalize(restored))
            recog_loss = class_loss_fn(pred, labels)

            #  FINAL LOSS (lambda applied here)
            loss = img_loss + lambda_recog * recog_loss

            # ----- Backprop -----
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

            pbar.set_postfix(
                total=loss.item(),
                img=img_loss.item(),
                cls=recog_loss.item(),
                lam=lambda_recog
            )

        print(f"Epoch {epoch+1} Loss: {total_loss/len(loader):.4f}")

    # SAVE MODEL FOR EACH LAMBDA
    torch.save(model.state_dict(), f"RA_resnet18_lambda_{lambda_recog}.pth")

    print(f"Saved model for λ = {lambda_recog}")


========== Training with λ = 0 ==========



100%|██████████| 375/375 [02:55<00:00,  2.13it/s, cls=0.567, img=0.00159, lam=0, total=0.00159]  


Epoch 1 Loss: 0.0018


100%|██████████| 375/375 [03:04<00:00,  2.04it/s, cls=1.17, img=0.00172, lam=0, total=0.00172]   


Epoch 2 Loss: 0.0017


...

Epoch 3 Loss: 0.7356


100%|██████████| 375/375 [02:29<00:00,  2.51it/s, cls=0.705, img=0.00191, lam=1, total=0.707]


Epoch 4 Loss: 0.7298


100%|██████████| 375/375 [02:32<00:00,  2.47it/s, cls=0.399, img=0.00214, lam=1, total=0.401]  

Epoch 5 Loss: 0.6978
Saved model for λ = 1


In [33]:
# torch.save(model.state_dict(),"RA_model.pth")
# print("Recognition-Aware model saved as RA_model.pth")
